In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

### What is Causal Inference?

**Causal Inference** is the process of drawing conclusions about cause-and-effect relationships from data.

> **The Golden Rule:** *"Correlation does not imply causation."*  
> Causal Inference is the set of tools we use to prove causation when we can't run a perfect controlled experiment.

In a perfect world, to know if a new website design (Treatment) increases sales (Outcome), we would need to see:

- **What happened** when the user saw the new design (**Factual**)
- **What would have happened** if that same user saw the old design at the exact same time (**Counterfactual**)

> **The Fundamental Problem:** We can **never** observe the counterfactual. A single user can only be in one state at a time.


| What we see | What we need | Problem |
|-------------|--------------|---------|
| User A saw new design → converted | What if they saw old design? | ❌ Unknown |
| User B saw old design → didn't convert | What if they saw new design? | ❌ Unknown |
| User C saw new design → didn't convert | What if they saw old design? | ❌ Unknown |

**We can only observe ONE reality per user!**

| Concept | Meaning | Why It's a Problem |
|---------|---------|-------------------|
| **Factual** | What actually happened | We can observe this ✅ |
| **Counterfactual** | What would have happened otherwise | We can NEVER observe this ❌ |

**We can never truly know if the treatment caused the outcome or not!**

Since we can't observe the counterfactual for a single user, we:

1. **Compare groups** instead of individuals
2. **Randomize** to make groups similar
3. **Assume** the groups are exchangeable


In [2]:
###

### Business Analogy
Imagine you are a CEO deciding whether to give your employees a **$5,000 bonus (Treatment)** to increase **productivity (Outcome)**.

- **Factual:** You give the bonus to Employee A. Their productivity goes up by 10%.
- **Counterfactual:** What would Employee A's productivity have been if you didn't give them the bonus? **You will never know.**


> **So, how do we solve this?** We use statistics to estimate the counterfactual by finding a **"control group"** that acts as a stand-in for what the treated group would have looked like if they hadn't been treated.

In [ ]:
###

### The Three Conditions for Causality
For a relationship to be causal, you must satisfy these three conditions:

| Condition | Definition | ML Equivalent |
|-----------|------------|---------------|
| **1. Association** | $X$ and $Y$ must be related | Your ML model must find a relationship (Easy! ✅) |
| **2. Temporal Precedence** | $X$ must happen before $Y$ | Feature must precede the target in time (Often violated in ML! ⚠️) |
| **3. No Confounding** | There is no third variable $Z$ that causes both $X$ and $Y$ | Your ML model doesn't care about this, but causality REQUIRES it ❌ |

### Why ML Models Can't Do Causality (On Their Own)

In [8]:
# Scenario: Do painkillers (X) reduce headaches (Y)?
# Confounder (Z): Headache severity.

np.random.seed(42)
n = 1000

# generate the Confounder (Severity of headache)
severity = np.random.normal(5, 1, n)  # Scale 1-10

# severity causes people to TAKE painkillers (X)
# (people with worse headaches are more likely to take the pill)
takes_pill = np.random.binomial(1, 1 / (1 + np.exp(5 - severity)), n)

# severity causes the OUTCOME (headache relief)
# (worse headaches take longer to go away, even with pills)
relief = 5 - 0.5 * severity + 2 * takes_pill + np.random.normal(0, 0.5, n)

df = pd.DataFrame({
    'Took_Pill': takes_pill,
    'Relief_Score': relief,
    'Severity': severity
})

print(df.head())
print()

# --- ML Approach (Correlation) ---
model = LinearRegression()
model.fit(df[['Took_Pill']], df['Relief_Score'])
print("ML (Correlation) Result:")
print(f"  Coefficient for taking pill: {model.coef_[0]:.2f}")
print(f"  Interpretation: Taking the pill INCREASES relief by {model.coef_[0]:.2f} points.")

# --- Causal Approach (Controlling for Confounder) ---
causal_model = LinearRegression()
causal_model.fit(df[['Took_Pill', 'Severity']], df['Relief_Score'])
print("\nCausal (Adjusted) Result:")
print(f"  {causal_model.coef_}")
print(f"  Coefficient for taking pill (holding severity constant): {causal_model.coef_[0]:.2f}")
print(f"  Interpretation: When we account for severity, the pill only adds {causal_model.coef_[0]:.2f} points.")
print(f"  The ML model was CONFOUNDED by severity!")

   Took_Pill  Relief_Score  Severity
0          1      4.097038  5.496714
1          0      2.193054  4.861736
2          1      4.335743  5.647689
3          1      4.408710  6.523030
4          0      1.679490  4.765847

ML (Correlation) Result:
  Coefficient for taking pill: 1.63
  Interpretation: Taking the pill INCREASES relief by 1.63 points.

Causal (Adjusted) Result:
  [ 2.01315819 -0.47408107]
  Coefficient for taking pill (holding severity constant): 2.01
  Interpretation: When we account for severity, the pill only adds 2.01 points.
  The ML model was CONFOUNDED by severity!


ML thought:  Pill = 1.63 points of relief ❌

Truth:       Pill = 2.01 points of relief ✅

Severity was confounding the relationship!

**ML models find correlations. Causal inference finds the true effect by controlling for confounders!**

### How We Solve This?

These are the **"Big Four"** tools for solving causal inference problems:

| Tool | What it does | Best used when... |
|------|--------------|-------------------|
| **1. Randomized Controlled Trials (A/B Tests)** | Randomly assign treatment/control | You have full control and budget ✅ |
| **2. Adjustment (Regression)** | Statistically "hold confounders constant" | You know all the confounders ✅ |
| **3. Instrumental Variables** | Use a "natural lottery" to randomize treatment | You have an instrument (e.g., distance to hospital) ✅ |
| **4. Difference-in-Differences** | Compare trends before/after treatment | You have pre/post data ✅ |